In [1]:
# imports
import mne
import numpy as np
import os

In [4]:
# load basis index of file
def load_eeg_data(file_path: str) -> mne.io.BaseRaw:
    """
    Load EEG data using the appropriate reader based on file extension.
    Parameters :
        file_path(str): Path to the EEG file
    Returns :
        raw (mne.io.BasRaw): Raw object loaded with preload=True
    Raises:
        ValueError: If the file extension is not supported
        FileNotFoundError: If the file does not exist at the given path
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f'File not found at path:{file_path}')
    # Extract extension
    ext = os.path.splitext(file_path)[-1].lower()

    #Map each extension to its correspondong mne.io reader
    reader_map ={
        '.edf' : mne.io.read_raw_edf,           # European Data Format
        '.bdf' : mne.io.read_raw_bdf,           # BioSemi Data Format
        '.vhdr' : mne.io.read_raw_brainvision,  # BrainVision
    }
    if ext not in reader_map:
        raise ValueError(
            f"Extensin {ext} is not supported. "
            f"Supported formats:{list(reader_map.keys())}"
        )
    # Load file preload=True transfers all data into RAM immediately
    raw = reader_map[ext](file_path, preload=True)
    return raw

# Extract and display metadat
def explore_metadata(raw: mne.io.BaseRaw) -> dict:
    """
    Extracts and prints key metadata from a Raw object.

    Parameters:
        raw: Loaded Raw object

    Returns:
        dict: Dictionary of extracted metadata
    """
    info = raw.info 
    # Extract values
    fs = info['sfreq'] # Sampling frequency (HZ)
    n_ch = info['nchan'] # Total number of channels
    ch5 = info['ch_names'][:5] # Firts 5 channel names
    N = raw.n_times # Total number of time samples
    T = N / fs # signal duration T = N / fs
    # Structured display
    sep ="-"*45
    print("  EEG Signal Metadata")
    print(sep)
    print(f"  Sampling frequency  (f_s) : {fs:.1f} Hz")
    print(f"  Number of channels        : {n_ch}")
    print(f"  First 5 channel names     : {ch5}")
    print(f"  Total samples         (N) : {N}")
    print(f"  Signal duration       (T) : {T:.3f} s")
    print(sep)
    print(f"  Formula: T = N / f_s  →  {N} / {fs} = {T:.3f} s")
    print(sep)

    return {
        'fs': fs, 'n_channels': n_ch,
        'first_5_channels': ch5, 'n_samples': N, 'duration_s': T
    }
# Entry point
if __name__ == "__main__":
    local_file = 'test.edf'

    try:
        raw = load_eeg_data(local_file)
        print(f"Local file'{local_file}'loaded successfully.")
    except (FileNotFoundError, ValueError):
        raw = load_sample_eegbci()
        print("EEGBCI sample data loaded successfully.")
    metadata = explore_metadata(raw)



Extracting EDF parameters from test.edf...
Setting channel info structure...
Creating raw.info structure...


C:\Users\Surface Laptop 3\AppData\Local\Temp\ipykernel_3668\1090849013.py:30: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = reader_map[ext](file_path, preload=True)


Reading 0 ... 921599  =      0.000 ...  3599.996 secs...
Local file'test.edf'loaded successfully.
  EEG Signal Metadata
---------------------------------------------
  Sampling frequency  (f_s) : 256.0 Hz
  Number of channels        : 23
  First 5 channel names     : ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3']
  Total samples         (N) : 921600
  Signal duration       (T) : 3600.000 s
---------------------------------------------
  Formula: T = N / f_s  →  921600 / 256.0 = 3600.000 s
---------------------------------------------
